# Parsing completo dos XMLs

In [5]:
import os
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm import tqdm
import json
import html
import re

# --------------------------------------------------
# Caminhos
# --------------------------------------------------
XML_DIR = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\XML Dataset\xml_raw"
RANKS_PATH = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\boardgames_ranks.csv"
OUTPUT_XLSX = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\XML Dataset\bgg_metadata_final.xlsx"
OUTPUT_JSON = r"C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\XML Dataset\bgg_metadata_final.json"

# --------------------------------------------------
# Funções auxiliares
# --------------------------------------------------

def clean_illegal_chars(text):
    """Remove caracteres de controlo ilegais para Excel (0x00–0x1F exceto \t, \n, \r)."""
    if isinstance(text, str):
        return re.sub(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]", "", text)
    return text

def safe_text(elem):
    """Extrai texto limpo e remove entidades HTML + caracteres ilegais."""
    if elem is None or elem.text is None:
        return ""
    text = html.unescape(elem.text.strip())
    return clean_illegal_chars(text)

def parse_geek_type(item):
    """Extrai o 'Type' (Strategy, Family, etc.) a partir dos ranks de família."""
    ranks = item.findall(".//statistics/ratings/ranks/rank[@type='family']")
    best = None
    best_val = None
    for r in ranks:
        val = r.get("value")
        try:
            val_num = int(val)
        except (TypeError, ValueError):
            continue
        friendly = r.get("friendlyname", "")
        if friendly.endswith(" Game Rank"):
            gtype = friendly.replace(" Game Rank", "")
        else:
            nm = r.get("name", "")
            gtype = nm.replace("games", "").replace("boardgame", "").strip().title()
        if best_val is None or val_num < best_val:
            best_val = val_num
            best = gtype
    return best or ""

def parse_xml(filepath):
    """Converte um ficheiro XML do BGG num dicionário estruturado."""
    try:
        root = ET.parse(filepath).getroot()

        # Encontrar o primeiro <item> válido
        item = None
        for it in root.findall("item"):
            if it.get("id") and it.get("type"):
                item = it
                break
        if item is None:
            return None

        gid = item.get("id", "")
        thing_type = item.get("type", "")  # boardgame / expansion / accessory
        name_tag = item.find("name[@type='primary']")
        name = name_tag.get("value") if name_tag is not None else ""
        desc = safe_text(item.find("description"))

        cats = [lnk.get("value") for lnk in item.findall("link[@type='boardgamecategory']")]
        mechs = [lnk.get("value") for lnk in item.findall("link[@type='boardgamemechanic']")]
        fams = [lnk.get("value") for lnk in item.findall("link[@type='boardgamefamily']")]
        impl = [lnk.get("value") for lnk in item.findall("link[@type='boardgameimplementation']")]

        geek_type = parse_geek_type(item)

        return {
            "id": int(gid) if str(gid).isdigit() else gid,
            "name": clean_illegal_chars(name),
            "description": desc,
            "type": clean_illegal_chars(thing_type),
            "geek_type": clean_illegal_chars(geek_type),
            "categories": clean_illegal_chars(", ".join([c for c in cats if c])),
            "mechanisms": clean_illegal_chars(", ".join([m for m in mechs if m])),
            "families": clean_illegal_chars(", ".join([f for f in fams if f])),
            "implements": clean_illegal_chars(", ".join([i for i in impl if i])),
        }
    except Exception:
        return None

# --------------------------------------------------
# Carregar IDs na ordem de popularidade (boardgames_ranks.csv)
# --------------------------------------------------
df_ranks = pd.read_csv(RANKS_PATH)
ordered_ids = df_ranks["id"].dropna().astype(int).tolist()

# Indexar XMLs disponíveis por ID
xml_index = {
    int(fn[:-4]): os.path.join(XML_DIR, fn)
    for fn in os.listdir(XML_DIR)
    if fn.endswith(".xml") and fn[:-4].isdigit()
}

print(f"XMLs encontrados: {len(xml_index):,}")

# --------------------------------------------------
# Parsing na ordem de ranking
# --------------------------------------------------
rows = []
for gid in tqdm(ordered_ids, desc="A processar XMLs por ranking"):
    fp = xml_index.get(gid)
    if not fp:
        continue
    rec = parse_xml(fp)
    if rec:
        rows.append(rec)

df = pd.DataFrame(rows)
print(f"Registos válidos: {len(df):,}")

# --------------------------------------------------
# Limpeza global final (segurança extra)
# --------------------------------------------------
df = df.applymap(clean_illegal_chars)

# --------------------------------------------------
# Guardar ficheiros
# --------------------------------------------------
df.to_excel(OUTPUT_XLSX, index=False)
df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

print("\nDataset final criado com sucesso e limpo de caracteres ilegais.")
print(f"Excel: {OUTPUT_XLSX}")
print(f"JSON:  {OUTPUT_JSON}")


XMLs encontrados: 166,971


A processar XMLs por ranking: 100%|██████████| 167041/167041 [02:06<00:00, 1316.97it/s]


Registos válidos: 166,942


C:\Users\marco\AppData\Local\Temp\ipykernel_9056\1327427320.py:130: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_illegal_chars)



Dataset final criado com sucesso e limpo de caracteres ilegais.
Excel: C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\XML Dataset\bgg_metadata_final.xlsx
JSON:  C:\Users\marco\OneDrive\Ambiente de Trabalho\Tese\Dataset\XML Dataset\bgg_metadata_final.json
